In [7]:
import csv
import numpy as np

from src.data_collection.file.file_paths import FilePaths
from src.data_collection.request.champion_loader import load_champion_json
from src.data_collection.request.mastery_loader import get_mastery_data

In [8]:
class Champion:
    def __init__(self, name, matrix_position):
        self.name = name
        self.matrix_position = matrix_position

## Load data and map champion ID to Name and alphabetical index

In [9]:
champion_data = load_champion_json()
id_champ_map = {
    int(champion_id): Champion(champion_data[champion_id], i)
    for i, champion_id in enumerate(champion_data.keys())
}

champion_list = [champion.name for champion in id_champ_map.values()]

## Save data in matrix format for analysis and df

In [15]:
data = get_mastery_data(FilePaths.mastery_directory())
user_ids = []

mastery_matrix = np.zeros((len(data.keys()), len(champion_list)))

for user_idx, key in enumerate(data.keys()):
    current_mastery = data[key]
    user_ids.append(key)
    for nth_most_popular, entry in enumerate(current_mastery):
        if nth_most_popular == 0:
            maximum = entry['championPoints']
        mastery_matrix[user_idx][id_champ_map[entry['championId']].matrix_position] = entry['championPoints']/maximum

user_id_rows = [[user_id] for user_id in user_ids]
fields = ['player_id']
with open(FilePaths.viz_player_ids_file(), 'w') as f:
    write = csv.writer(f)
    write.writerow(fields)
    write.writerows(user_id_rows)

np.savetxt(FilePaths.mastery_matrix_file(), mastery_matrix, delimiter=',', header=','.join(champion_list))

unique_user_ids = set(user_ids)
assert(len(user_ids) == unique_user_ids.__len__())

## Save data in surprise format to load into surprise
The dataset has 5326 users from the ranked pool. For each user, the dataset contains the user's 25 most played champions and each champion's corresponding mastery score for the user. Champion mastery score is highly correlated with playtime.

I loaded in champion mastery data and set each user's highest played champion's mastery score to 1. I then divide every other champion's mastery point score by the highest played champion's mastery point score. We assume that any champion after the 25th has a rating of 0. This is a crude approximation since the mean of the 25th highest mastery score was 0.09. The rating scale is therefore 0 to 1.

In [5]:
class Score:
    def __init__(self, index):
        self.rank = index
        self.count = 0
        self.total = 0

In [6]:
data = get_mastery_data(FilePaths.mastery_directory())
ratings = [] # list of items
NTH_HIGHEST = [1, 2, 4, 8, 16, 25, 50]
average_scores = {n - 1: Score(n) for n in NTH_HIGHEST}

for i, key in enumerate(data.keys()):
    for rank, entry in enumerate(data[key]):
        if rank == 0:
            # sorted in descending order
            maximum = entry['championPoints']
        if rank + 1 in NTH_HIGHEST:
            average_scores[rank].total += entry['championPoints']/maximum
            average_scores[rank].count += 1

    for n in NTH_HIGHEST:
        if rank + 1 < n:
            average_scores[n - 1].count += 1

for score in average_scores.values():
    print(f'Mean of {score.rank}th highest champion score: {score.total/score.count}')


Mean of 1th highest champion score: 1.0
Mean of 2th highest champion score: 0.6255137288485725
Mean of 4th highest champion score: 0.39213098524687584
Mean of 8th highest champion score: 0.23911801988350623
Mean of 16th highest champion score: 0.13653984342884248
Mean of 25th highest champion score: 0.08984883363075863
Mean of 50th highest champion score: 0.02922535656982236
